# ONERA CFD Dataset — Exploratory Analysis

**Objetivo:** Caracterizar el dataset de simulaciones CFD del proyecto *Shock Detection via Physics-Informed ML* para fundamentar las elecciones de diseño del paper.

**Dataset:**
- `X_train.npy` / `X_test.npy` — 9 features de entrada (Mach, AoA, Pi, x, y, z, nx, ny, nz)
- `Ytrain.npy` / `Ytest.npy` — 4 targets aerodinámicos (Cp, Cfx, Cfy, Cfz)
- `X_train_derived.npy` / `X_test_derived.npy` — 19 features (9 originales + 10 derivados)
- `dataset.csv` — metadatos de las 468 configuraciones de simulación

**Tamaños:** ~81M puntos de train, ~40M de test

In [ ]:
import sys
sys.path.insert(0, '..')  # acceder a config y src desde notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

DATA_DIR = Path('../data')
print('Data dir:', DATA_DIR.resolve())

## 1. Tamaños y shapes del dataset

In [ ]:
files = {
    'X_train':         DATA_DIR / 'X_train.npy',
    'X_test':          DATA_DIR / 'X_test.npy',
    'Y_train':         DATA_DIR / 'Ytrain.npy',
    'Y_test':          DATA_DIR / 'Ytest.npy',
    'X_train_derived': DATA_DIR / 'X_train_derived.npy',
    'X_test_derived':  DATA_DIR / 'X_test_derived.npy',
}

print(f"{'File':<22} {'Shape':>20}  {'Size (GB)':>10}")
print('-' * 56)
for name, path in files.items():
    if path.exists():
        arr = np.load(path, mmap_mode='r')
        size_gb = arr.nbytes / 1e9
        print(f"{name:<22} {str(arr.shape):>20}  {size_gb:>10.2f}")
    else:
        print(f"{name:<22} {'NOT FOUND':>20}")

## 2. Metadatos de simulaciones (dataset.csv)

In [ ]:
df = pd.read_csv(DATA_DIR / 'dataset.csv')
print(f'Simulaciones: {len(df)}')
print(f'Columnas: {list(df.columns)}')
df.head()

In [ ]:
df.describe().T.style.format('{:.3f}')

In [ ]:
# Espacio Mach–AoA cubierto por las simulaciones
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

train_df = df[df['Train'] == 1] if 'Train' in df.columns else df
test_df  = df[df['Train'] == 0] if 'Train' in df.columns else pd.DataFrame()

ax = axes[0]
ax.scatter(train_df['Mach'], train_df['AoA'], alpha=0.5, s=20, label='Train')
if not test_df.empty:
    ax.scatter(test_df['Mach'], test_df['AoA'], alpha=0.7, s=20, marker='^', label='Test')
ax.set_xlabel('Mach'); ax.set_ylabel('AoA (°)')
ax.set_title('Espacio de diseño Mach–AoA')
ax.legend(); ax.grid(True, alpha=0.3)

# Distribuciones Mach y AoA
ax = axes[1]
ax.hist(df['Mach'], bins=30, alpha=0.6, label='Mach', color='steelblue')
ax2 = ax.twinx()
ax2.hist(df['AoA'], bins=30, alpha=0.4, label='AoA', color='orange')
ax.set_xlabel('Valor'); ax.set_ylabel('N sim (Mach)', color='steelblue')
ax2.set_ylabel('N sim (AoA)', color='orange')
ax.set_title('Distribución de condiciones de vuelo')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2)

plt.tight_layout()
plt.savefig('../outputs/plots/eda_mach_aoa_space.png', bbox_inches='tight')
plt.show()

## 3. Distribución de features de entrada (X, muestra 200K puntos)

In [ ]:
# Cargar muestra
X_raw = np.load(DATA_DIR / 'X_train.npy', mmap_mode='r')
Y_raw = np.load(DATA_DIR / 'Ytrain.npy',  mmap_mode='r')

SAMPLE = 200_000
idx = np.random.choice(len(X_raw), SAMPLE, replace=False)
X_s = np.asarray(X_raw[idx], dtype=np.float32)
Y_s = np.asarray(Y_raw[idx], dtype=np.float32)

feature_names = ['Mach', 'AoA', 'Pi', 'x', 'y', 'z', 'nx', 'ny', 'nz']
print(f'Muestra: {X_s.shape}')

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()

for i, name in enumerate(feature_names):
    ax = axes[i]
    data = X_s[:, i]
    ax.hist(data, bins=80, color='steelblue', alpha=0.75, edgecolor='none')
    ax.axvline(np.mean(data), color='red',    lw=1.5, linestyle='--', label=f'μ={np.mean(data):.3f}')
    ax.axvline(np.median(data), color='orange', lw=1.5, linestyle=':',  label=f'med={np.median(data):.3f}')
    ax.set_title(name); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('Distribución de features de entrada (X_train, 200K muestra)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../outputs/plots/eda_input_distributions.png', bbox_inches='tight')
plt.show()

# Tabla de estadísticas
stats = pd.DataFrame({
    'mean':   X_s.mean(axis=0),
    'std':    X_s.std(axis=0),
    'min':    X_s.min(axis=0),
    'p25':    np.percentile(X_s, 25, axis=0),
    'p50':    np.percentile(X_s, 50, axis=0),
    'p75':    np.percentile(X_s, 75, axis=0),
    'max':    X_s.max(axis=0),
}, index=feature_names)
stats.style.format('{:.4f}')

## 4. Distribución de targets aerodinámicos (Y)

In [ ]:
target_names = ['Cp', 'Cfx', 'Cfy', 'Cfz']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, name in enumerate(target_names):
    ax = axes[i]
    data = Y_s[:, i]
    # Recortar percentiles extremos para visualización
    lo, hi = np.percentile(data, 1), np.percentile(data, 99)
    mask = (data >= lo) & (data <= hi)
    ax.hist(data[mask], bins=100, color='coral', alpha=0.75, edgecolor='none')
    ax.axvline(np.mean(data), color='navy', lw=2, linestyle='--', label=f'μ={np.mean(data):.4f}')
    ax.set_title(name); ax.legend(); ax.grid(True, alpha=0.3)
    ax.set_xlabel(name)

plt.suptitle('Distribución de outputs aerodinámicos (Y_train, 200K muestra)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../outputs/plots/eda_output_distributions.png', bbox_inches='tight')
plt.show()

pd.DataFrame({
    'mean':  Y_s.mean(axis=0),
    'std':   Y_s.std(axis=0),
    'min':   Y_s.min(axis=0),
    'p5':    np.percentile(Y_s, 5, axis=0),
    'p95':   np.percentile(Y_s, 95, axis=0),
    'max':   Y_s.max(axis=0),
}, index=target_names).style.format('{:.4f}')

## 5. Features derivados — indicadores de choque

In [ ]:
derived_path = DATA_DIR / 'X_train_derived.npy'
if not derived_path.exists():
    print('X_train_derived.npy no encontrado. Ejecuta: python preprocess_data.py')
else:
    X_der = np.load(derived_path, mmap_mode='r')
    X_der_s = np.asarray(X_der[idx], dtype=np.float32)
    print(f'Derived features shape: {X_der_s.shape}')

    derived_names = [
        'M_local', 'grad_p', 'cp_loss', 'shock_indicator',
        'Cf_mag', 'q_dyn', 'Pi_norm', 'AoA_norm', 'grad_cf', 'L_factor'
    ]

    fig, axes = plt.subplots(2, 5, figsize=(18, 7))
    axes = axes.flatten()

    for i, name in enumerate(derived_names):
        ax = axes[i]
        data = X_der_s[:, 9 + i]
        lo, hi = np.percentile(data, 1), np.percentile(data, 99)
        mask = (data >= lo) & (data <= hi)
        color = 'darkorange' if name == 'shock_indicator' else 'mediumpurple'
        ax.hist(data[mask], bins=80, color=color, alpha=0.75, edgecolor='none')
        ax.set_title(name, fontsize=9)
        ax.grid(True, alpha=0.3)

    plt.suptitle('Distribución de features derivados (200K muestra)', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig('../outputs/plots/eda_derived_distributions.png', bbox_inches='tight')
    plt.show()

## 6. Indicador de choque — análisis de umbral

In [ ]:
if 'X_der_s' in dir():
    shock_ind = X_der_s[:, 12]  # DERIVED_FEATURE_INDICES['shock_indicator'] = 12
    cf_mag    = X_der_s[:, 13]  # DERIVED_FEATURE_INDICES['Cf_mag'] = 13

    # Binarización por z-score (umbral = 0.5σ)
    shock_norm = (shock_ind - shock_ind.mean()) / (shock_ind.std() + 1e-8)
    threshold  = 0.5
    shock_label = (shock_norm > threshold).astype(int)

    print(f'Puntos en choque (shock_indicator > {threshold}σ): '
          f'{shock_label.sum():,} / {len(shock_label):,} ({shock_label.mean()*100:.1f}%)')

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Distribución del indicador de choque
    ax = axes[0]
    ax.hist(shock_norm, bins=100, color='steelblue', alpha=0.7, edgecolor='none')
    ax.axvline(threshold, color='red', lw=2, linestyle='--', label=f'umbral={threshold}')
    ax.set_xlabel('shock_indicator (normalizado)'); ax.set_ylabel('Frecuencia')
    ax.set_title('Distribución shock_indicator')
    ax.legend(); ax.grid(True, alpha=0.3)

    # Cp en zonas de choque vs no choque
    ax = axes[1]
    cp = Y_s[:, 0]
    lo, hi = np.percentile(cp, 1), np.percentile(cp, 99)
    ax.hist(cp[(shock_label == 0) & (cp >= lo) & (cp <= hi)],
            bins=80, alpha=0.6, color='steelblue', label='No choque', density=True)
    ax.hist(cp[(shock_label == 1) & (cp >= lo) & (cp <= hi)],
            bins=80, alpha=0.6, color='red', label='Choque', density=True)
    ax.set_xlabel('Cp'); ax.set_ylabel('Densidad')
    ax.set_title('Cp: choque vs no choque')
    ax.legend(); ax.grid(True, alpha=0.3)

    # M_local en zonas de choque
    ax = axes[2]
    m_local = X_der_s[:, 9]
    lo, hi = np.percentile(m_local, 1), np.percentile(m_local, 99)
    ax.hist(m_local[(shock_label == 0) & (m_local >= lo) & (m_local <= hi)],
            bins=80, alpha=0.6, color='steelblue', label='No choque', density=True)
    ax.hist(m_local[(shock_label == 1) & (m_local >= lo) & (m_local <= hi)],
            bins=80, alpha=0.6, color='red', label='Choque', density=True)
    ax.axvline(1.0, color='k', lw=1.5, linestyle=':', label='M=1')
    ax.set_xlabel('M_local'); ax.set_ylabel('Densidad')
    ax.set_title('M_local: choque vs no choque')
    ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('../outputs/plots/eda_shock_analysis.png', bbox_inches='tight')
    plt.show()

## 7. Distribución espacial — geometría de la superficie

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

x, y, z_coord = X_s[:, 3], X_s[:, 4], X_s[:, 5]

# Colorear por Cp
cp_norm = (Y_s[:, 0] - Y_s[:, 0].mean()) / Y_s[:, 0].std()
cp_clip = np.clip(cp_norm, -3, 3)

subsample = 50_000
idx2 = np.random.choice(len(x), subsample, replace=False)

sc = axes[0].scatter(x[idx2], y[idx2], c=cp_clip[idx2], cmap='RdBu_r', s=0.5, alpha=0.6)
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')
axes[0].set_title('Vista xy — coloreado por Cp')
plt.colorbar(sc, ax=axes[0], label='Cp (norm)')

sc = axes[1].scatter(x[idx2], z_coord[idx2], c=cp_clip[idx2], cmap='RdBu_r', s=0.5, alpha=0.6)
axes[1].set_xlabel('x'); axes[1].set_ylabel('z')
axes[1].set_title('Vista xz — coloreado por Cp')
plt.colorbar(sc, ax=axes[1], label='Cp (norm)')

# Scatter M_local vs Cp
if 'X_der_s' in dir():
    m_local = X_der_s[idx2, 9]
    lo, hi = np.percentile(m_local, 1), np.percentile(m_local, 99)
    mask = (m_local >= lo) & (m_local <= hi)
    sc2 = axes[2].scatter(m_local[mask], Y_s[idx2][mask, 0],
                          c=cp_clip[idx2][mask], cmap='RdBu_r', s=0.5, alpha=0.5)
    axes[2].axvline(1.0, color='k', lw=1.5, linestyle=':', label='M_local=1')
    axes[2].set_xlabel('M_local'); axes[2].set_ylabel('Cp')
    axes[2].set_title('M_local vs Cp (diagnóstico de choque)')
    axes[2].legend(); axes[2].grid(True, alpha=0.3)
    plt.colorbar(sc2, ax=axes[2], label='Cp (norm)')

plt.tight_layout()
plt.savefig('../outputs/plots/eda_spatial_distribution.png', bbox_inches='tight')
plt.show()

## 8. Matriz de correlaciones

In [ ]:
# Correlaciones entre inputs originales y outputs
all_data = np.hstack([X_s, Y_s])
all_names = feature_names + target_names
corr = np.corrcoef(all_data.T)

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(all_names))); ax.set_xticklabels(all_names, rotation=45, ha='right')
ax.set_yticks(range(len(all_names))); ax.set_yticklabels(all_names)
plt.colorbar(im, ax=ax, label='Correlación de Pearson')
ax.set_title('Matriz de correlaciones X ∪ Y')

# Anotar valores
for i in range(len(all_names)):
    for j in range(len(all_names)):
        ax.text(j, i, f'{corr[i,j]:.2f}', ha='center', va='center',
                fontsize=7, color='black' if abs(corr[i,j]) < 0.7 else 'white')

plt.tight_layout()
plt.savefig('../outputs/plots/eda_correlation_matrix.png', bbox_inches='tight')
plt.show()

## 9. Estadísticas de confianza de las simulaciones

In [ ]:
if 'confidence_weight_simple' in df.columns:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    axes[0].hist(df['confidence_weight_simple'], bins=30, color='teal', alpha=0.7)
    axes[0].set_title('Distribución confidence_weight'); axes[0].grid(True, alpha=0.3)

    axes[1].scatter(df['Mach'], df['confidence_weight_simple'],
                    c=df['AoA'], cmap='viridis', s=20, alpha=0.7)
    axes[1].set_xlabel('Mach'); axes[1].set_ylabel('confidence_weight')
    axes[1].set_title('Confianza vs Mach (color=AoA)')

    for col in ['std_drag', 'std_lift', 'std_mom']:
        if col in df.columns:
            axes[2].scatter(df['Mach'], df[col], label=col, alpha=0.6, s=15)
    axes[2].set_xlabel('Mach'); axes[2].set_ylabel('Std')
    axes[2].set_title('Variabilidad aerodinámica vs Mach')
    axes[2].legend(); axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('../outputs/plots/eda_confidence.png', bbox_inches='tight')
    plt.show()
else:
    print('Columna confidence_weight_simple no encontrada en dataset.csv')

## 10. Resumen para el paper

Tabla de estadísticas key para la sección de *Dataset* del paper.

In [ ]:
summary = {
    'Total training points':  f'{len(X_raw):,}',
    'Total test points':      f'{len(np.load(DATA_DIR / "X_test.npy", mmap_mode="r")):,}',
    'Input features (raw)':   9,
    'Input features (derived)': 19,
    'Output targets':         4,
    'Simulations':            len(df),
    'Mach range':             f'{df["Mach"].min():.2f} – {df["Mach"].max():.2f}',
    'AoA range (°)':          f'{df["AoA"].min():.2f} – {df["AoA"].max():.2f}',
    'Shock ratio (%)':        f'{shock_label.mean()*100:.1f}' if 'shock_label' in dir() else 'N/A',
}

summary_df = pd.DataFrame.from_dict(summary, orient='index', columns=['Value'])
print('\n=== Dataset Summary for Paper ===')
print(summary_df.to_string())
summary_df